# Partition-lattice constants $M_r$, $B_r$, and the charge-counting recurrence

<a href="https://colab.research.google.com/github/kootru-repo/charge-filtered-cumulant-residuals/blob/main/notebooks/01_partition_constants.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200"/></a>


Manuscript reference: Section III, Theorem 1, and Lemma 1 (charge-counting recurrence).

Definitions:
- $M_r = \max_{m \le r} \sum_{\pi \in \Pi_m} (|\pi|-1)!$
- $B_r = \max_{3 \le m \le r} \sum_{\pi:\, \exists |B|>2} M_r^{|\pi|-1}$
- $B^{\mathrm{charge}}_r(W) = (P_{h,z}(M_r) - Q_{h,z}(M_r)) / M_r$

Manuscript values verified here: $M_3 = 6$, $M_4 = 26$, $M_5 = 150$, $B_4 = 105$, $B_5 = 227{,}251$, $B^{\mathrm{charge}}_4(a^\dagger_i a_j n_k n_\ell) = 53$.

In [ ]:
# Bootstrap on Colab / fresh env (skipped if package + data are already present).
# Local users who cloned the repo and ran 'uv sync' skip the install entirely.
# On Colab the bootstrap clones the repo so data/ files are reachable via
# notebook-relative paths, chdir's into it, then installs the package with uv.
import importlib, importlib.util as _ilu
import pathlib as _pl
_HAS_PKG = _ilu.find_spec("connected_layer_sector") is not None
_HAS_DATA = _pl.Path("data").is_dir() or _pl.Path("../data").is_dir()
if not (_HAS_PKG and _HAS_DATA):
    import subprocess as _sp, os as _os
    _REPO = "/content/charge-filtered-cumulant-residuals"
    if not _pl.Path(_REPO).is_dir():
        _sp.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/kootru-repo/charge-filtered-cumulant-residuals.git",
            _REPO,
        ])
    _os.chdir(_REPO)
    # Pin uv to a known directory. UV_INSTALL_DIR is the installer's explicit
    # override; without it the location depends on CARGO_HOME / XDG_BIN_HOME
    # inheritance (Colab differs from local Linux).
    _UV_DIR = "/tmp/uv-bootstrap"
    _os.makedirs(_UV_DIR, exist_ok=True)
    _env = {**_os.environ, "UV_INSTALL_DIR": _UV_DIR}
    _sp.check_call(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, executable="/bin/bash", env=_env,
    )
    _UV = f"{_UV_DIR}/uv"
    # Non-editable install: lands directly in site-packages (already on sys.path),
    # so the import below works without sys.path gymnastics. Editable installs
    # rely on .pth files Python only reads at startup.
    _sp.check_call([_UV, "pip", "install", "--system", "-q", "."])
    importlib.invalidate_caches()  # flush find_spec cache so re-runs idempotently short-circuit

import connected_layer_sector
print(f"connected_layer_sector {connected_layer_sector.__version__} ready")


In [1]:
from connected_layer_sector import M_r_const, B_r_const, set_partitions
from connected_layer_sector.constants import (
    B_charge_r,
    charge_filtered_polynomial,
    evaluate_polynomial,
)
import math

## $M_r$ table

In [2]:
for r in (3, 4, 5):
    print(f"M_{r} = {M_r_const(r):.0f}")

assert M_r_const(3) == 6.0
assert M_r_const(4) == 26.0
assert M_r_const(5) == 150.0

M_3 = 6
M_4 = 26
M_5 = 150


## $B_r$ table

In [3]:
for r in (3, 4, 5):
    print(f"B_{r} = {B_r_const(r):.0f}")

assert B_r_const(3) == 1.0
assert B_r_const(4) == 105.0
assert B_r_const(5) == 227251.0

B_3 = 1
B_4 = 105
B_5 = 227251


## Direct enumeration: $B_4 = \max\big(D_3(M_4),\, D_4(M_4)\big)$

$B_r$ is a max over $3 \le m \le r$ of $D_m(M_r)$, where $D_m(x) = \sum_{\pi \in \Pi_m,\, \exists |B|>2} x^{|\pi|-1}$ enumerates partitions of $[m]$ with at least one block of size $>2$. For $r = 4$:

- $D_3(x) = 1$ (the single partition $\{1,2,3\}$ of $[3]$ has $|\pi|-1 = 0$, contributing $x^0 = 1$). So $D_3(M_4) = 1$.
- $D_4(x) = 1 + 4x$ (one all-in-one partition contributing $x^0 = 1$, plus four $1{+}3$ partitions each contributing $x^1$). So $D_4(M_4) = 1 + 4 \cdot 26 = 105$.

The max is $\max(1, 105) = 105$. The cell below confirms the $D_4$ enumeration directly; $D_3$ is computed inline for completeness.

In [ ]:
def D_m_enumerate(m):
    """Enumerate partitions of [m] with at least one block of size > 2,
    grouped by |pi|. Returns (counts_by_pi_size, polynomial_coefficients)
    where the polynomial is in the order [coef of x^0, coef of x^1, ...]."""
    by_count = {}
    for pi in set_partitions(range(m)):
        if any(len(b) > 2 for b in pi):
            by_count[len(pi)] = by_count.get(len(pi), 0) + 1
    max_k = max(by_count) if by_count else 0
    coefs = [by_count.get(k, 0) for k in range(1, max_k + 1)]
    return by_count, coefs

# D_3: the only partition of [3] with a block of size > 2 is {1,2,3}, so D_3(x) = 1.
by_count_3, coefs_3 = D_m_enumerate(3)
D3_at_M4 = sum(c * (M_r_const(4) ** k) for k, c in enumerate(coefs_3))
print(f"D_3 coefficients (low to high): {coefs_3}  ->  D_3(M_4 = 26) = {D3_at_M4}")
assert by_count_3 == {1: 1}, f"unexpected D_3 enumeration: {by_count_3}"

# D_4: 1 all-in-one + 4 (1+3) partitions = 1 + 4x.
by_count_4, coefs_4 = D_m_enumerate(4)
D4_at_M4 = sum(c * (M_r_const(4) ** k) for k, c in enumerate(coefs_4))
print(f"D_4 coefficients (low to high): {coefs_4}  ->  D_4(M_4 = 26) = {D4_at_M4}")
print()
print("partitions of [4] with a block of size > 2, by |pi|:")
for k, n in sorted(by_count_4.items()):
    print(f"  |pi| = {k}: {n} partition(s) -> contributes {n} * M_4^{k - 1}")
assert by_count_4 == {1: 1, 2: 4}, f"unexpected D_4 enumeration: {by_count_4}"

print()
print(f"B_4 = max(D_3(M_4), D_4(M_4)) = max({D3_at_M4}, {D4_at_M4}) = {max(D3_at_M4, D4_at_M4)}")
assert max(D3_at_M4, D4_at_M4) == 105
assert B_r_const(4) == 105.0

## Charge-counting recurrence (Lemma 1) — worked example

Manuscript: for $W = a^\dagger_i a_j n_k n_\ell$ with $(h, z) = (1, 2)$:

$$P_{1,2}(x) = x + 3x^2 + x^3, \qquad Q_{1,2}(x) = x^2 + x^3$$

and at $r = 4$ (so $M_4 = 26$):

$$B^{\mathrm{charge}}_4(W) = (P_{1,2}(26) - Q_{1,2}(26)) / 26 = (19630 - 18252) / 26 = 53.$$

In [5]:
P, Q = charge_filtered_polynomial(1, 2)
print("P_{1,2}(x) coefficients (low to high degree):", P)
print("Q_{1,2}(x) coefficients (low to high degree):", Q)

P26 = evaluate_polynomial(P, 26.0)
Q26 = evaluate_polynomial(Q, 26.0)
print(f"\nP_{{1,2}}(26) = {P26:.0f} (manuscript: 19630)")
print(f"Q_{{1,2}}(26) = {Q26:.0f} (manuscript: 18252)")
print(f"B^charge_4(W) = ({P26:.0f} - {Q26:.0f}) / 26 = {B_charge_r(1, 2, 4):.0f}")

assert P == [0.0, 1.0, 3.0, 1.0]
assert Q == [0.0, 0.0, 1.0, 1.0]
assert math.isclose(B_charge_r(1, 2, 4), 53.0, abs_tol=1e-9)
assert math.isclose(B_charge_r(1, 2, 5), 301.0, abs_tol=1e-9)

P_{1,2}(x) coefficients (low to high degree): [0.0, 1.0, 3.0, 1.0]
Q_{1,2}(x) coefficients (low to high degree): [0.0, 0.0, 1.0, 1.0]

P_{1,2}(26) = 19630 (manuscript: 19630)
Q_{1,2}(26) = 18252 (manuscript: 18252)
B^charge_4(W) = (19630 - 18252) / 26 = 53


## Sanity-check assertions

Headline manuscript values reproduced numerically (run all asserts in this notebook):

In [6]:
assert (M_r_const(3), M_r_const(4), M_r_const(5)) == (6.0, 26.0, 150.0)
assert (B_r_const(3), B_r_const(4), B_r_const(5)) == (1.0, 105.0, 227251.0)
assert B_charge_r(1, 2, 4) == 53.0
assert B_charge_r(1, 2, 5) == 301.0
print("PASS: all manuscript closed-form values match.")

PASS: all manuscript closed-form values match.
